# Accessing and reusing public data from PANGAEA with `pangaeapy` 
#### By: Johanna Jäger  
#### Last modified: 04. March 2026
This project was carried out as part of the data science module for marine biologists at the University of Rostock (WiSe 25/26).

## 1. Motivation and research question

In order to contextualize data sets of any kind, observational data is required. Nevertheless, the process of establishing links between disparate data sets (molecular, ecological, etc.) and existing observations is an intricate one. That demands a suitable database with an extensive collection, wherein precise searches can be executed.  
One such database is [PANGAEA](https://www.pangaea.de/). It is a digital library that houses a collection of Earth and environmental data. PANGAEA provides curated metadata for data comprehensibility and dataset DOIs for accessibility. This is intended to ensure the FAIRness of the data sets, a principle to guarantee equitable access and utilisation of data. Data stored in long-term archives should be FAIR (findable, accessible, interoperable, reusable), but how interoperable and reusable are they in reality?

With this in mind, the following objectives were set for this project:

- Finding an easily modifiable and targeted search strategy for PANGAEA datasets
- Creating an automated workflow (as far as possible) that works for all parameters and locations without having to change a lot of code
- Applying some basic methods to inspect the downloaded data and check its reusability

[This code](https://github.com/Data-Science-Center-UB/Python-From-PANGAEA-to-Interactive-Maps/blob/main/Notebooks_Solutions/01_download_pangaea_data.ipynb) was used as a basis for the workflow and was modified. 
## 2. Preparation
### 2.1 Import of Packages
First the installation of `pangaeapy` is required, the Python package for downloading datasets and metadata. The official Documentation can be found [here](https://pangaea-data-publisher.github.io/pangaeapy/user_guide.html).

In [1]:
!pip install -q pangaeapy

Import of `pangaeapy`:

In [2]:
import pangaeapy as pan
from pangaeapy.pandataset import PanDataSet # for the download of metadata

Import of other relevant Python packages:

In [3]:
import sys, os, contextlib
import pandas as pd
import numpy as np
import requests 
from urllib.request import urlopen, urlretrieve

Check the version of used packages for reproducibility:

In [4]:
!pip install -q print_versions
from print_versions import print_versions

print_versions(globals())

pangaeapy==1.1.0
pandas==2.3.3
numpy==2.3.5
requests==2.32.5


To ignore warnings for a more clean notebook:

In [5]:
import warnings
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=(SettingWithCopyWarning))
warnings.simplefilter(action='ignore', category=FutureWarning)

## 3. PANGAEA Search Query
The search function is used to find matching data records. It opens up a range of search options, including free text with wildcards, searching for keywords in specific fields (e.g. project, author or parameter), and the use of [Boolean expressions](https://libguides.usc.edu/searching/boolean) like AND, OR, NOT. The documentation and rules of PANGAEA search can be found [here](https://wiki.pangaea.de/wiki/PANGAEA_search).

### 3.1 Define Query

The objective of this project is to download the Pangaea data sets for a specific parameter in a region. To do this, the according variables must be defined in the search query: 
1. Defining the query with a geographical limitation (bounding box), using the example of the Baltic Sea. The coordinates for the boundary are in the following order: Minimum Longitude (Western boundary), Minimum Latitude (Southern Boundary), Maximum Longitude (Eastern Boundary), Maximum Latitude (Northern Boundary).
2. Parameter search is possible in many ways: A simple free-text search, or with `parameter:oxygen` or `@param754`, the id. In the second option the unit is undefined and the query will show all results with any oxygen parameter. For more comparable results, it's better to use the exact id, as the unit is defined.  
Let's look at the difference it makes in query results:

In [6]:
# free-text search of "oxygen"
query1 = pan.PanQuery('oxygen', bbox=(9, 53, 31, 66), limit=500)
print(f'There are {query1.totalcount} query results for "oxygen".')

# search for all parameters with the name "oxygen"
query2 = pan.PanQuery('parameter:oxygen', bbox=(9, 53, 31, 66), limit=500)
ratio1 = query2.totalcount / query1.totalcount
print(f'There are {query2.totalcount} query results for "parameter:oxygen" ({ratio1:.2%} of Query1).')

# search for the specific parameter oxygen [µmol/l] with ID:754 
query3 = pan.PanQuery('@param754', bbox=(9, 53, 31, 66), limit=500)
ratio2 = query3.totalcount / query1.totalcount
print(f'There are {query3.totalcount} query results for "@param754" ({ratio2:.2%} of Query1).')
print("-"*40)

There are 3351 query results for "oxygen".
There are 2627 query results for "parameter:oxygen" (78.39% of Query1).
There are 361 query results for "@param754" (10.77% of Query1).
----------------------------------------


Different parameters can be looked up in the extensive [parameter list](https://www.pangaea.de/lists/parameter/all-byname), the most common ones are:  
* oxygen [µmol/l], id: 754
* salinity, id: 716
* water temperature [°C], id: 717
* pH, id: 764
* chlorophyll a [µg/l], id: 2136

It's also possible to search for a specific publishing year (e.g. 2010) with `year:2010`. Searching for datasets published since 2010 is not easily possible, as every year has to be spelled out: `AND (year:2010 OR year:2011 OR year:2012 ...)`.

### 3.2 Get Query results

<div class="alert alert-block alert-info">
<b>Note:</b> This part differs greatly from the reference code because there was a problem with the PanQuery limit. Instead of downloading the results in chunks of 500, the command kept iterating through the first 500 results. I fixed it by looping over the results in steps of 500, skipping the already retrieved pages and appending them to a dataframe. 
</div>

In [7]:
# Define the location and parameter
bbox = (9, 53, 31, 66)
id = 754

# Run the query 
query = pan.PanQuery(f'@param{id}', bbox=bbox, limit=500) # PanQuery can only retrieve 500 datasets at once (maxed out)

# Print total number of results
print(f'There are {query.totalcount} query results.')
print("-"*40)

# Create empty dataframe to collect all pages
df_query_results_all = pd.DataFrame()

# Loop over all results in steps of 500, breaking the result in smaller chunks
for offset in np.arange(0, query.totalcount, 500):

    # Create a new query for each page using the same parameters
    qs = pan.PanQuery(query.query,      # reuse the query string
                      bbox=bbox,        # reuse bbox (as PanQuery does not store it)
                      limit=500,
                      offset=offset)    # fetch results in pages of 500

    # Convert this page of results to a dataframe
    df_qs = pd.DataFrame(qs.result)

    # Append to the full dataframe
    df_query_results_all = pd.concat([df_query_results_all, df_qs],
                                     ignore_index=True)

There are 361 query results.
----------------------------------------


Perform a *sanity check* to ensure there are no duplicates (meaning the looping error was fixed):

In [8]:
print(f'There are {len(df_query_results_all['URI'].unique())} unique entries.')
print("-"*40)

There are 361 unique entries.
----------------------------------------


### 3.3 Save query results 
After formulating the search, the query should be saved. To do this, directories for data and metadata storage must be defined. If they do not already exist, they will be created.  
The query and metadata will be stored in `data_directory`.  
The datasets will be stored in `dataset_directory`.

<div class="alert alert-block alert-info">
<b>Note:</b> It is important to name the folder according to the parameter in the interest of maintaining order and enabling automatic file naming. It's important to change the directory name when changing the search query, otherwise the files will be overwritten.
</div>

In [9]:
data_directory = "../Pangaea_Project/oxygen"
dataset_directory = "../Pangaea_Project/oxygen/datasets"

# Creating directories if they don't exist
if not os.path.isdir(data_directory):
    os.mkdir(data_directory)

if not os.path.isdir(dataset_directory):
    os.mkdir(dataset_directory)

# extracting the parameter name for automatic file naming
parameter_name = os.path.basename(os.path.dirname(os.path.normpath(dataset_directory)))


Save the query results as a tab-delimited text file. 

In [10]:
df_query_results_all.to_csv(os.path.join(data_directory, "PANGAEA_query.txt"), 
                            encoding="utf-8", 
                            sep="\t", 
                            index=False)

## 4. Get metadata
Metadata can be understood as a description of the data itself, and is important for the management and understanding of data. It provides a comprehensive overview and is imperative for troubleshooting (e.g. measurement errors) and subsequent citations using DOIs and ORCID identifiers.
### 4.1 Download metadata
The metadata is downloaded for each data record by iterating over all query results. There is a lot of metadata that can be fetched, but not all of it is always useful. For the purpose of this project, the device is an important piece of information because it influences the parameter measurements. 

<div class="alert alert-block alert-info">
<b>Note:</b> Almost all metadata query options are listed in the code. If different metadata is required, these options can be activated by uncommenting them. It is important to check the output table with some text options because they can change the format (e.g. if tab stops are built into 'comment').  
    This step may take a while, depending on the amount of metadata requested. 
</div>

In [11]:
# context manager to silence stderr (hides "Data set is protected..." spam)
@contextlib.contextmanager
def suppress_stderr():
    with open(os.devnull, 'w') as devnull:
        old_stderr = sys.stderr
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stderr = old_stderr

# start of the progress update: total number of query results
total = len(df_query_results_all)
# counting the iterations and updating the progress
for i, (ind, value) in enumerate(df_query_results_all['URI'].items(), start=1):

    # print progress update
    sys.stdout.write(f"\rProcessed {i}/{total}")
    sys.stdout.flush()

    with suppress_stderr():
        try:
            ds = PanDataSet(id=value, include_data=False)

            # core identifiers
            df_query_results_all.loc[ind,'ID'] = ds.id
            df_query_results_all.loc[ind,'dataset URI'] = ds.uri
            df_query_results_all.loc[ind,'dataset DOI'] = ds.doi
            df_query_results_all.loc[ind,'dataset title'] = str(ds.title)   

            # extended attributes
            df_query_results_all.loc[ind,'abstract'] = str(ds.abstract)
            df_query_results_all.loc[ind,'year'] = ds.year
            df_query_results_all.loc[ind,'citation'] = ds.citation
            df_query_results_all.loc[ind,'collection members'] = ', '.join(ds.collection_members)
            df_query_results_all.loc[ind,'isCollection'] = "Yes" if ds.isCollection else "No"
            df_query_results_all.loc[ind,'publication date'] = ds.date
            df_query_results_all.loc[ind,'projects'] = ds.projects
            df_query_results_all.loc[ind,'mintimeextent'] = ds.mintimeextent
            df_query_results_all.loc[ind,'maxtimeextent'] = ds.maxtimeextent
            df_query_results_all.loc[ind,'mean latitude'] = ds.geometryextent.get("meanLatitude")
            # df_query_results_all.loc[ind,'min latitude'] = ds.geometryextent.get("southBoundLatitude")
            # df_query_results_all.loc[ind,'max latitude'] = ds.geometryextent.get("northBoundLatitude")
            df_query_results_all.loc[ind,'mean longitude'] = ds.geometryextent.get("meanLongitude")
            # df_query_results_all.loc[ind,'min longitude'] = ds.geometryextent.get("westBoundLongitude")
            # df_query_results_all.loc[ind,'max longitude'] = ds.geometryextent.get("eastBoundLongitude")
            # df_query_results_all.loc[ind,'comment'] = str(ds.comment)
            # df_query_results_all.loc[ind,'relations'] = ds.relations
            # df_query_results_all.loc[ind,'supplement_to'] = ds.supplement_to
            # df_query_results_all.loc[ind,'keywords'] = str(ds.keywords)
            # df_query_results_all.loc[ind,'licence'] = ds.licence
            # df_query_results_all.loc[ind,'auth_token'] = ds.auth_token

            # authors
            df_query_results_all.loc[ind,'first author fullname'] = ds.authors[0].fullname if ds.authors else None
            df_query_results_all.loc[ind,'all authors fullnames'] = "; ".join([x.fullname for x in ds.authors])
            df_query_results_all.loc[ind,'all authors orcids'] = "; ".join([x.ORCID if x.ORCID else "no ORCID" for x in ds.authors])

            # parameters
            df_query_results_all.loc[ind,'parameters'] = "; ".join(
                [f'{param.name} [{param.unit}]' if param.unit else param.name for param in ds.params.values()]
            )

            # events and devices
            campaign_names = {event.campaign.name for event in ds.events if event.campaign and event.campaign.name}
            df_query_results_all.loc[ind, 'campaign'] = "; ".join(campaign_names)

            events = ds.getEventsAsFrame()
            if "device" in events.columns:
                devices = set(events["device"].fillna("no device"))
                df_query_results_all.loc[ind, "device"] = "; ".join(devices)
            else:
                df_query_results_all.loc[ind, "device"] = "no device info"

            # status requests
            # df_query_results_all.loc[ind,'curationlevel'] = ds.curationlevel
            # df_query_results_all.loc[ind,'processinglevel'] = ds.processinglevel
            # df_query_results_all.loc[ind,'moratorium'] = ds.moratorium
            # df_query_results_all.loc[ind,'loginstatus'] = ds.loginstatus
            # df_query_results_all.loc[ind,'cache_expiry_days'] = ds.cache_expiry_days
            # df_query_results_all.loc[ind,'cachedir'] = ds.cachedir
            # df_query_results_all.loc[ind,'datastatus'] = ds.datastatus
            # df_query_results_all.loc[ind,'registrystatus'] = ds.registrystatus
            # df_query_results_all.loc[ind,'lastupdate'] = ds.lastupdate

            # quality flags
            # df_query_results_all.loc[ind,'quality_flags'] = ds.quality_flags
            # df_query_results_all.loc[ind,'qcdata'] = ds.qcdata

        except Exception as e:
             # store error message
             df_query_results_all.loc[ind, 'error'] = str(e)

Processed 361/361

Show the first three lines of metadata to get a sense of the format. Should an issue arise, it could be seen in this section.

In [12]:
df_query_results_all.head(3)[['ID', 'dataset title', 'year', 'device']]

,ID,dataset title,year,device
0,898738.0,Composition and vertical flux of particulate o...,2019,no device info
1,967477.0,Gypsum flocculation experiment in river Paimio...,2025,NaN
2,950878.0,Incubation experiment of Fucus vesiculosus and...,2022,no device info


### 4.2 Save metadata
Save the metadata in a text file. The file name is dynamic and changes with the parameter.

In [13]:
df_query_results_all.to_csv(os.path.join(data_directory, f'PANGAEA_{parameter_name}_{id}_metadata.txt'),
                            encoding="utf-8",
                            sep="\t",
                            index=False)

print(f'PANGAEA metadata saved')

PANGAEA metadata saved


## 5. Get datasets
Translating the parameter names to long names with unit. 

In [14]:
def get_long_parameters(ds):
    """Translate short parameters names to long names including unit

    Args:
        ds (PANGAEA dataset): PANGAEA dataset
    """
    ds.data.columns =  [f'{param.name} [{param.unit}]' if param.unit else param.name for param in ds.params.values()]

### 5.1 Download datasets
In this step the actual datasets are downloaded. The code iterates over the query results and loads each dataset in a `pandas` dataframe and stores them in a python dictionary. If the dataset is restricted and not accessible, it will be skipped.

In [15]:
# context manager to silence stderr (hides "Data set is protected..." spam)
@contextlib.contextmanager
def suppress_stderr():
    with open(os.devnull, 'w') as devnull:
        old_stderr = sys.stderr
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stderr = old_stderr

# dictionary for downloaded datasets
data_dict = {}

# counters
downloaded_count = 0
skipped_count = 0
total = len(df_query_results_all['URI'])

for i, pangaea_doi in enumerate(df_query_results_all['URI'], start=1):
    with suppress_stderr():
        try:
            ds = PanDataSet(pangaea_doi, enable_cache=True)

            # skip restricted or collection datasets
            if ds.isCollection or getattr(ds, "restricted", False) or ds.data is None or ds.data.empty:
                skipped_count += 1
                continue

            # rename parameters safely
            if len(ds.data.columns) == len(ds.params):
                ds.data.columns = [
                    f'{param.name} [{param.unit}]' if param.unit else param.name
                    for param in ds.params.values()
                ]

            # extract numeric ID from DOI
            pangaea_id = pangaea_doi.split('A.')[1]

            # store dataset in dictionary
            data_dict[pangaea_id] = ds.data
            downloaded_count += 1

        except Exception:
            skipped_count += 1

    # progress update
    percent_skipped = (skipped_count / i * 100) if i > 0 else 0
    
    sys.stdout.write(
    f"\rProcessed {i}/{total} | "
    f"Downloaded: {downloaded_count} | "
    f"Skipped: {skipped_count} ({percent_skipped:.1f}%)"
    )
    sys.stdout.flush()

print("\n" + "-"*40)

Processed 361/361 | Downloaded: 281 | Skipped: 80 (22.2%)
----------------------------------------


It is evident that a significant proportion of certain parameters are restricted or skipped. They are not available, mostly because they are in the moratorium period. This means that it may be an ongoing project or that a related manuscript is currently being reviewed. 
### 5.2 Save datasets
Loop over each dataset in the dictionary and save them to `dataset_directory`. The file name is dynamic and changes with the parameter.

In [16]:
from IPython.display import clear_output

# save individual datasets of the dictionary
total = len(data_dict)

for i, (key, df) in enumerate(data_dict.items(), start=1):
    df.to_csv(
        os.path.join(dataset_directory, f'PANGAEA_{parameter_name}_{id}_dataset_{key}.txt'),
        index=False,
        sep="\t",
        encoding="utf-8"
    )
    # single updating line
    sys.stdout.write(f"\rProcessed {i}/{total} datasets")
    sys.stdout.flush()

print("\nAll datasets saved.")

Processed 281/281 datasets
All datasets saved.


Perform a _sanity check_ to ensure that the numbers are correct and that everything has worked as intended.

In [17]:
print("Total DOIs:", len(df_query_results_all['URI']))
print("Downloaded with metadata:", downloaded_count)
print("Saved with data:", len(data_dict))
print("Skipped (restricted/collection):", skipped_count)

Total DOIs: 361
Downloaded with metadata: 281
Saved with data: 281
Skipped (restricted/collection): 80


## Next steps
Now that the metadata and datasets themselves are retrieved from PANGAEA, the next steps for data processing and visualization are done in R Studio. The follow-up script is `Pangaea_data_2.Rmd`.